# Experiment 3: fitness-dependent movement

In [1]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
from pyvirtualdisplay import Display

In [2]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output_experiment_three_fitness_dependent_movement'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'
parameter_file.write_text('0.2 0.02\n')

9

In [3]:
display = Display(visible=False, size=(1200, 900))
display.start()

In [4]:
runner = ra.runAUTO()

try:
    eq_forward = ac.run(e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200)
    eq_backward = ac.run(DS='-', runner=runner, NMX=4000, NPR=200)
    eq = (eq_forward + eq_backward).relabel()
    ac.save(eq, 'eq')

    bp_curve = ac.run(
        eq('BP1'),
        ICP=[28, 31],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 31: [0.0, 4.0]},
        runner=runner,
    ).relabel()
    ac.save(bp_curve, 'bp_curve')

    lp_curve = ac.run(
        eq('LP1'),
        ICP=[28, 31],
        ISW=2,
        DS=1.0e-3,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: [-0.25, 0.5], 31: [0.0, 4.0]},
        runner=runner,
    ).relabel()
    ac.save(lp_curve, 'lp_curve')

    codim2 = (bp_curve + lp_curve).relabel()
    ac.save(codim2, 'codim2')
finally:
    runner.config(clean=True)
    ac.clean()

gfortran -g -fopenmp -O -c common_model.f90 -o common_model.o
gfortran -g -fopenmp -O common_model.o -o common_model.exe /auto/lib/*.o
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.51996E+00   0.00000E+00   7.10303E+00   0.00000E+00   0.00000E+00   6.33849E+00   0.00000E+00
   1     9  BP    2   1.13281E-01   9.51996E+00  -3.09856E-21   7.10303E+00  -2.15447E-21  -3.12760E-21   6.33849E+00  -2.27193E-21
   1    17  UZ    3   5.00000E-01   9.51996E+00  -4.46299E-32   7.10303E+00  -4.20077E-32  -5.94312E-32   6.33849E+00  -4.52338E-32

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2    41  MX    4   1.30266E-01   8.10975E+00   2.58576E-01   6.04945E+00   2.02776E-01   2.60798E-01   5.38059E+00   2.13143E-01

  BR    PT  TY  LAB       mu         L2-NO

KeyError: 'Label LP1 not found'

In [ ]:
p = ac.plot('eq', hide=True)
p.config(
    stability=True,
    grid=False,
    bifurcation_x=['mu'],
    bifurcation_y=['PL'],
    xlabel='mu',
    ylabel='PL',
    title='',
    minx=0.0,
    maxx=0.1,
)
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_1d.png'))
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_1d.svg'))

In [ ]:
p = ac.plot('codim2', hide=True)
p.config(
    grid=False,
    bifurcation_x=['beta'],
    bifurcation_y=['mu'],
    xlabel='beta',
    ylabel='mu',
    title='',
    minx=0.0,
    maxx=4.0,
    miny=0.0,
    maxy=0.1,
)
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_2d.png'))
p.savefig(str(output_folder / 'experiment_three_fitness_dependent_movement_2d.svg'))

In [ ]:
display.stop()
parameter_file.unlink(missing_ok=True)
ac.delete('eq')
ac.delete('bp_curve')
ac.delete('lp_curve')
ac.delete('codim2')